# Solutions — Context

Only look here after you've actually tried the exercises in `context.ipynb`.

### LESSON 55 — Exercise

**1. The count.** **Five** components take `user` as a prop — `Layout`, `Page`, `Header`,
`Toolbar`, `Avatar`. **One** reads a field from it: `Avatar`, which uses `user.name` and
`user.initials`. Four out of five exist only to forward it.

**2. Adding a `theme`.** Ten lines touched — a parameter and a prop in each of the five
components — to deliver one string to one place. The cost is linear in depth and multiplies by
the number of values, which is the whole argument of the lesson.

**3. The `children` version.**

```jsx
function Layout({ children }) { return <main>{children}</main>; }
function Page({ children })   { return <section>{children}<p>…</p></section>; }
function Header({ children }) { return <header><h2>Dashboard</h2>{children}</header>; }

export default function Experiment25() {
  const user = { name: "Ada Lovelace", initials: "AL" };
  return (
    <Layout>
      <Page>
        <Header>
          <Toolbar user={user} />
        </Header>
      </Page>
    </Layout>
  );
}
```

Now **two** components mention `user`: `Toolbar` and `Avatar`. Three layers stopped knowing
about it entirely — not by routing around them, but because they are no longer between the data
and its user.

**4. What `children` costs.** The tree moves into the parent. `Experiment25` now shows the whole
nesting, which is more to read in one place, and the layout components lose the ability to
decide *what* goes inside them — they only decide where it goes. You would not do this when the
intermediate component genuinely owns its content (a `ProductCard` that decides what a card
contains), or when the nesting would become so deep that the parent turns into a wall of JSX.

It is a trade, not a strict improvement: explicit structure at the top in exchange for ignorant,
reusable layers underneath.

**Common mistakes.**

- Counting "how many lines" rather than "how many components now depend on this". The coupling
  is the cost, not the typing.
- Reaching for Context before trying `children`. It is LESSON 5 and it removes layers rather
  than bypassing them.
- Thinking explicit props are a failure. React's own docs call them a feature: "The person
  maintaining your code will be glad you've made the data flow explicit with props."

### LESSON 55 — Mini challenge

| | answer | why |
|---|---|---|
| 1. signed-in user across three screens | **Context** | many readers, many depths, rarely changes |
| 2. `posts` through `Layout` → `Sidebar` → `PostList` | **`children`** | the layers are visual and should not be between the data and its user |
| 3. theme read by ~30 components | **Context** | the canonical case |
| 4. `productId` to three sections of one page | **pass props** | one level, three siblings — explicit is clearer and cheaper |
| 5. `dispatch` to a dozen components | **Context** | many readers at depth, and the value never changes identity |
| 6. `selectedRow` for a table and its neighbour | **pass props** | lift to the common parent (LESSON 29) — the distance is one level |

**The two that look identical: 2 and 4.** Both are a value passed through or across a small
number of layers, and they get different answers.

What distinguishes them is **whether the components in between have a reason to exist without
the data**. In 2, `Layout` and `Sidebar` are visual shells that do not care about posts — so
they should take `children` and get out of the way. In 4, the three sections *are* about the
product; passing `productId` to them is describing what they are for, not routing around them.

The test: if removing the data from a component's signature would leave it a perfectly sensible
component, it should not have been in the middle. If it would leave it meaningless, the prop
belongs there.

### LESSON 56 — Exercise

In [ ]:
// 1. Clicking "switch user" re-renders App and everything below it - Layout, Page, Header,
//    Toolbar and Avatar all log. Of those, only Avatar mentions `user` in its code.
//
//    The gap: the middle components re-render because their PARENT re-rendered (LESSON 38),
//    not because they read the context. Context did not save them a render; it saved them
//    from KNOWING anything. Those are different benefits, and only the second one was ever
//    promised.
//
// 2. Changing the outer user leaves the nested avatar as "Katherine Johnson (KJ)". Measured:
//    outer goes Ada -> Grace, nested stays Katherine. The rule is that a component reads the
//    NEAREST provider above it, and a nearer provider shadows a further one for its subtree.
//
// 3. Deleting the nested <ThemeContext> wrapper: the nested Avatar still gets a theme, because
//    the OUTER ThemeContext is still above it in the tree. Context passes through everything
//    in between, including the <hr /> and the <p>. It would only fall back to the default if
//    there were no provider above it at all.
//
// 4. With a default of { name: "Guest", initials: "G" }, an Avatar outside both providers
//    renders "Guest (G)" instead of crashing.
//
//    A default like that is a GOOD idea when there is a genuinely sensible neutral value -
//    "light" for a theme, "en" for a locale, a no-op function. It is a BAD idea when the
//    absence of a provider is a bug: a `null` default makes that bug crash immediately and
//    obviously, whereas a "Guest" default renders a plausible page and hides the wiring
//    mistake until a user asks why they are logged out.
//
// 5. Swapping to <UserContext.Provider value={...}> changes nothing on screen and produces no
//    console message - measured. Both forms work in React 19.3.0. Which one to write is
//    decided by direction of travel, not by behaviour: React has said it will deprecate
//    `.Provider` in a future version, so new code uses <UserContext value={...}>.

console.log("context passes through everything in between");

**Common mistakes.**

- Expecting Context to prevent re-renders of the middle components. It removes their knowledge,
  not their renders.
- Calling `useContext` inside a condition or after an early return (LESSON 37).
- Creating the context inside a component, so a new one is made every render and no provider
  ever matches.
- Using a friendly default to paper over a missing provider.

### LESSON 56 — Mini challenge

**1. A breaks the Rules of Hooks** (LESSON 37): `useContext` is called *after* a conditional
`return`. When `isLoggedIn` is false the component makes zero Hook calls, and when it is true it
makes one — so the Hook count changes between renders. React's message is *"Rendered fewer hooks
than expected. This may be caused by an accidental early return statement."* The fix is ordering:
call `useContext` above the `if`.

**2. B has no provider.** `useContext(ThemeContext)` with no `ThemeContext` above it returns the
**default value passed to `createContext`** — `"light"`. Nothing crashes, nothing warns, and the
`theme` state in `App` is simply never connected to anything: `setTheme` updates a value no
component reads. Missing is the provider — `<ThemeContext value={theme}>` around `<Layout />`.

This is the failure mode a friendly default creates, which is why question 4 of the exercise
matters.

**3. C's trap:** `value={{ user, setUser }}` builds a **new object on every render of `App`**.
Context values are compared by identity, so every reader re-renders whenever `App` renders —
even when `user` has not changed. LESSON 57 covers it, and the first fix is structural, not
memoisation.

### LESSON 57 — Exercise

In [ ]:
// 1. Combining into one AppContext: Avatar now re-renders when EITHER the user or the theme
//    changes, because it reads one context whose value changed. With two contexts it also
//    re-rendered in this experiment - but only because its parent re-rendered. The difference
//    becomes visible the moment a reader sits under a parent that did NOT re-render: with
//    split contexts it stays still, with a combined one it does not.
//
//    In the Profiler this shows as more components in each commit than the change justifies.
//
// 2. <UserContext value={{ user }}>: toggling the THEME re-renders App, which builds a new
//    { user } object, which is a new context value by Object.is (LESSON 40), so every reader
//    of UserContext re-renders - despite the user being identical. The object literal, not
//    the data, caused it.
//
// 3. State and dispatch in separate contexts:
//
//      <TasksContext value={state}>
//        <TasksDispatchContext value={dispatch}>
//          <Layout />
//        </TasksDispatchContext>
//      </TasksContext>
//
//    Why separate: `dispatch` is stable for the life of the component, so a component that
//    only dispatches never needs to re-render when the state changes. Combining them into
//    one object would give that component a new value every time the state moved, for no
//    reason - and dispatch-only components (buttons, forms) are usually the most numerous.
//
// 4. A value that should not be in Context here: the THEME, arguably - in this experiment it
//    has exactly one reader. With one reader at a known place, a prop is clearer and cheaper.
//    It belongs in Context in a real app because a real app has thirty readers; in this
//    experiment it is there to demonstrate the two-provider shape, which is worth being honest
//    about.

console.log("one provider per concern; split state from dispatch");

**Common mistakes.**

- One `AppContext` holding everything, because it looks tidier at the provider.
- Passing an object literal as the value and blaming Context for the re-renders.
- Reaching for memoisation before measuring, or before trying the structural fix.
- Treating Context as where state lives. It transports; components own.

### LESSON 57 — Mini challenge

**1. Why a cart change re-renders theme-only components.** There is one context value:
`{ user, theme, locale, cart, dispatch }`. Adding a cart item changes `cart`, which means the
provider builds a **new object** for the value, which by `Object.is` is a different value, which
means *every* component reading that context re-renders — all sixty — regardless of which field
they actually use. React has no way to know that a component only touched `.theme`; it sees one
context and one changed value.

**2. What to measure first.** Record the interaction in the **Profiler** (LESSON 35) and look at
which components appear in the commit and what each costs *relative to the others*. The
question is not "did sixty components render" but "is that where the time is going". It may not
be — the slowness could be in one expensive component, or in the work that adds the item.
LESSON 35's rule holds: measure before deciding.

**3. Two structural changes, in order.**

- **Split the context by concern.** `ThemeContext`, `UserContext`, `CartContext`,
  `LocaleContext`, and `dispatch` in its own. A cart change then wakes only cart readers. This
  is the largest win and requires no new API.
- **Separate state from dispatch.** Dispatch never changes identity, so components that only
  dispatch should read a context that never changes — and they are usually the most numerous.

Only after those, and only with a measurement, does memoising a context value make sense
(topic 23).

**4. Moving to Redux.** It would fix the re-render problem, because a store lets a component
subscribe to a *slice* of the state rather than to the whole value — that is a real,
architectural difference, and topic 27 covers it.

It would **not** fix anything else. The state still has to live somewhere and be shaped
sensibly; a badly organised store re-renders just as much as a badly organised context, and
Redux adds a library, a setup, and concepts the whole team must learn.

The question to ask first: **have we tried splitting the context?** If one `AppContext` is the
problem, four contexts solve it this afternoon with no new dependency. Reaching for a state
library to fix a structure problem usually moves the problem rather than removing it — and
topic 27 will make the case for a store on its own merits, not as an escape from this.